# 10. Bias in the data: heart-failure death

The same `ebdai` workflow as the Titanic demo, on the heart-failure table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025): 299 patients. The label `death` is death during follow-up. `sex` is 0 for a woman and 1 for a man.

On the Titanic the label gap was about 54 points and the rules copied it. Here about one patient in three dies in both groups, so there is almost no label gap to copy. Three questions:

1. Does the search still write a sentence that names `sex`?
2. If it does, does that sentence carry the decision? Read its DS.
3. Can a small gap in the predictions still appear?

> These rules come from a six-generation search on a few hundred patients. They are for reading a rule base, not for a decision about a patient.

On [Google Colab](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/10_bias_heart_failure.ipynb), run the setup cell first. If the next cell cannot import `ebdai`, restart the runtime and run every cell again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/10_bias_heart_failure.ipynb)


## Setup (Google Colab only)

Clones this repository and installs it. Skip it if `ebdai` is already installed (`pip install ebdai`, or `pip install -e .`).


In [ ]:
!git clone -q https://github.com/HAISymbiosis/EBD-AI.git
%cd EBD-AI
!pip install -q .


## The table, and the label gap

`load_heart_failure` stores the binary flags (`anaemia`, `diabetes`, `high_blood_pressure`, `sex`, `smoking`) as categories, so a rule can say `sex IS 0` (a woman) rather than `sex IS Low`. Numeric columns keep Low / Medium / High.

Before any model: 34 of 105 women die (32.4%) and 62 of 194 men die (32.0%). The label gap is about zero. A sentence can still name sex. The dominance score below says whether that sentence is doing the work. Equal rates here describe this recorded label and this recorded attribute in this sample.


In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_heart_failure, outcome_rates_by_group,
    plot_outcome_rates, plot_winning_rules_by_group, fairness_report,
    parse_printed_rules, winning_rules_by_group,
)

frame, sensitive = load_heart_failure()
X, y = features_and_target(frame, 'death')
print(X.dtypes)
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Death rate by sex')

age                         float64
anaemia                      object
creatinine_phosphokinase      int64
diabetes                     object
ejection_fraction             int64
high_blood_pressure          object
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                          object
smoking                      object
dtype: object
   group    n  n_positive  positive_rate
0      0  105          34       0.323810
1      1  194          62       0.319588


<Axes: title={'center': 'Death rate by sex'}, xlabel='Group', ylabel='Positive rate'>

## Fit, then read the rules and the gaps

The cell below is steps 2–4 of demo 09: split, fit, print the rules, predict, and count which rule won. `nRules=6`, and the same short search (`n_gen=6`, `pop_size=12`, `patience=3`). `ds_mode=1` fixes every weight at 1, so the strongest firing wins. Consequent 0 is survival. Consequent 1 is death.

In the stored run one rule carries the model: `IF serum_creatinine IS Low THEN 0`, with DS ≈ 0.54. One rule names sex:

```
IF anaemia IS 0 AND serum_sodium IS Medium AND sex IS 0 WITH DS 0.0061, ACC 1.0
```

`ACC 1.0` means it was right the few times it won. `DS 0.006` means it almost never wins, about ninety times less than the creatinine rule.

A gap is the women's rate minus the men's rate. On the test patients, 32 women and 67 men:

| What is counted | Women | Men | Gap |
| --- | --- | --- | --- |
| Death already in the full table | 34 of 105 | 62 of 194 | about 0 points |
| Death predicted | 3 of 32 | 2 of 67 | about 6 points |
| Deaths the model caught | 2 of 12 | 2 of 30 | about 10 points |
| False alarms, among those who lived | 1 of 20 | 0 of 37 | about 5 points |

Equalized odds prints the larger of the two error gaps, 0.10. The ratio prints 0 because the men's false-alarm rate is exactly zero: one false alarm against none, on 20 and 37 people. Read those two differences together with the counts.

The search does name `sex`. That sentence barely fires. A small prediction gap, 3 predicted deaths against 2, appears with almost no label gap behind it. The chart shows which sentence won for whom.


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=6, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))
counts = winning_rules_by_group(
    clf, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report or ''),
)
plot_winning_rules_by_group(counts, title='Winning heart-failure rules by sex')

------------
ACCURACY
Train performance: 0.795
Test performance: 0.6060606060606061
------------
MATTHEW CORRCOEF
Train performance: 0.4162463467383999
Test performance: 0.17534845441898042
------------
Rules for consequent: 0
----------------
IF serum_creatinine IS Low WITH DS 0.5408260303382613, ACC 0.7932960893854749, WGHT 1.0

Rules for consequent: 1
----------------
IF ejection_fraction IS Low WITH DS 0.06204445858071811, ACC 0.8181818181818182, WGHT 1.0
IF anaemia IS 0 AND serum_sodium IS Medium AND sex IS 0 WITH DS 0.006113775659428545, ACC 1.0, WGHT 1.0
IF creatinine_phosphokinase IS Low AND platelets IS Medium WITH DS 0.0215734818868575, ACC 0.7142857142857143, WGHT 1.0


   group   n  selection_rate       tpr   fpr       fnr   tnr
0      1  67        0.029851  0.066667  0.00  0.933333  1.00
1      0  32        0.093750  0.166667  0.05  0.833333  0.95
demographic_parity_difference    0.063899
demographic_parity_ratio         0.318408
equalized_odds_difference        0.100000
e

<Axes: title={'center': 'Winning heart-failure rules by sex'}, xlabel='Winning rule', ylabel='Share of group'>